# Apolos: YouVersion + Gloo AI Studio

This notebook provides reproducible API evidence for the Apolos hackathon submission. It uses the public Apolos backend for the licensed YouVersion catalog and can make an authenticated Gloo completion when Kaggle secrets are configured. Credentials are never printed.

In [ ]:
import os, requests

APOLOS_API = "https://apolos.io"
catalog = requests.get(
    f"{APOLOS_API}/api/youversion/versions",
    params={"language": "en", "popular": 0},
    timeout=20,
)
catalog.raise_for_status()
bibles = catalog.json()["data"]
print(f"Licensed English editions returned: {len(bibles)}")
[(b["id"], b["abbreviation"], b["localized_title"]) for b in bibles[:10]]

## Optional live Gloo verification

Add `GLOO_CLIENT_ID` and `GLOO_CLIENT_SECRET` as private Kaggle secrets. The cell exchanges them through OAuth2 client credentials and calls Completions V2. It reports only the generated text and model, never the credentials or bearer token.

In [ ]:
def kaggle_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)

client_id = kaggle_secret("GLOO_CLIENT_ID")
client_secret = kaggle_secret("GLOO_CLIENT_SECRET")

if not client_id or not client_secret:
    print("Skipped: configure private Kaggle Gloo secrets to run this cell.")
else:
    token_response = requests.post(
        "https://platform.ai.gloo.com/oauth2/token",
        auth=(client_id, client_secret),
        data={"grant_type": "client_credentials", "scope": "api/access"},
        timeout=20,
    )
    token_response.raise_for_status()
    access_token = token_response.json()["access_token"]

    completion = requests.post(
        "https://platform.ai.gloo.com/ai/v2/chat/completions",
        headers={"Authorization": f"Bearer {access_token}"},
        json={
            "model": "gloo-openai-gpt-5-mini",
            "auto_routing": False,
            "stream": False,
            "messages": [
                {"role": "system", "content": "Be concise, thoughtful, and grounded only in the supplied passage."},
                {"role": "user", "content": "Create one reflection question connecting Philippians 4:6-7 with anxiety about an uncertain future."},
            ],
        },
        timeout=60,
    )
    completion.raise_for_status()
    result = completion.json()
    print("Model:", result["model"])
    print(result["choices"][0]["message"]["content"])

## Production architecture

In the product, both integrations are server-side. The React/Tauri client calls Laravel; Laravel proxies and caches successful YouVersion responses, and exchanges Gloo credentials for a short-lived bearer token. Existing authentication, rate limits, and per-user AI budgets apply before inference.